# Gold Layer — dim_customer

Joins all customer-related silver tables (CRM + ERP) into a single `workspace.gold.dim_customer` dimension table.

---

## Table of Contents — Transformations Applied

| # | Transformation | Description |
|---|---|---|
| 1 | **Three-Way Join** | Joined CRM `customer_info` with ERP `customer_info1` (cust_az12) and ERP `customer_info2` (loc_a101) using SQL LEFT JOINs |
| 2 | **Direct Join** | All silver customer IDs are now in `AW000xxxxx` format, enabling direct equality joins on `customer_key` — no key transformation needed |
| 3 | **Unified Customer Dimension** | Combined CRM demographics (name, marital status, gender, create date) with ERP data (birth date, country) into a single dimension table |

---

## Source Tables

| Source Table | Layer | Columns Contributed |
|---|---|---|
| `workspace.silver.customer_info` | CRM | customer_id, customer_key, first_name, last_name, full_name, marital_status, gender, create_date |
| `workspace.silver.customer_info1` | ERP (cust_az12) | birth_date, gender (ERP source) |
| `workspace.silver.customer_info2` | ERP (loc_a101) | country |

---

## Join Logic

```
CRM customer_info (base, 18,494 rows)
  LEFT JOIN silver.customer_info2 (loc_a101)
    ON ci2.customer_id = ci.customer_key
  LEFT JOIN silver.customer_info1 (cust_az12)
    ON ci1.customer_id = ci.customer_key
```

| Metric | Count |
|---|---|
| Total Gold Rows | 18,494 |
| Matched with loc_a101 | 18,490 |
| Matched with cust_az12 | 18,490 |
| Unmatched (CRM only) | 4 |

---

## Implementation

All joins are implemented using **SQL** with LEFT JOIN to preserve all CRM customer records.

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.dim_customer
COMMENT 'Gold-layer customer dimension table joining CRM and ERP silver tables.'
AS
SELECT
    ci.customer_id,
    ci.customer_key,
    ci.first_name,
    ci.last_name,
    ci.full_name,
    ci1.birth_date,
    ci.gender,
    ci.marital_status,
    ci2.country,
    ci.create_date
FROM workspace.silver.customer_info ci
LEFT JOIN workspace.silver.customer_info2 ci2
    ON ci2.customer_id = ci.customer_key
LEFT JOIN workspace.silver.customer_info1 ci1
    ON ci1.customer_id = ci.customer_key

In [0]:
%sql
SELECT * FROM workspace.gold.dim_customer LIMIT 10